In [3]:
import math
from dataclasses import dataclass
from typing import Tuple, Optional, Literal

import torch
from torch import nn
import torch.nn.functional as F
import torch.distributed as dist

from kernel import act_quant, weight_dequant, fp8_gemm

In [4]:
world_size = 1  # 并行进程数
rank = 0  # 当前进程编号
block_size = 128  # 量化块大小
# 控制矩阵乘法（GEMM - General Matrix Multiply）的计算方式
gemm_impl: Literal["bf16", "fp8"] = "bf16"
# 控制多头潜在注意力（MLA - Multi-Head Latent Attention）的实现方式
attn_impl: Literal["naive", "absorb"] = "absorb"

In [5]:
# 模型参数
@dataclass
class ModelArgs:
    max_batch_size: int = 8
    max_seq_len: int = 4096 * 4
    dtype: Literal["bf16", "fp8"] = "bf16"
    vocab_size: int = 102400
    dim: int = 4096
    inter_dim: int = 11944
    moe_inter_dim: int = 1408
    n_layers: int = 27
    n_dense_layers: int = 1
    n_heads: int = 16
    # moe
    n_routed_experts: int = 64
    n_shared_experts: int = 2
    n_activated_experts: int = 6
    n_expert_groups: int = 1
    n_limited_groups: int = 1
    score_func: Literal["softmax", "sigmoid"] = "softmax"
    route_scale: float = 1.0
    # mla
    q_lora_rank: int = 0  # q
    kv_lora_rank: int = 512  # 
    qk_nope_head_dim: int = 128  # 
    qk_rope_head_dim: int = 64  # 
    v_head_dim: int = 128  # 
    # yarn
    original_seq_len: int = 4096
    rope_theta: float = 10000.0
    rope_factor: float = 40
    beta_fast: int = 32
    beta_slow: int = 1
    mscale: float = 1.0

In [6]:
class ParallelEmbedding(nn.Module):
    """支持并行的Embedding层"""
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        assert vocab_size % world_size == 0, f"Vocabulary size must be divisible by world size (world_size={world_size})"
        self.part_vocab_size = (vocab_size // world_size)
        self.vocab_start_idx = rank * self.part_vocab_size
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if world_size > 1:
            # 取出当前进程负责范围内的Embedding
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)
            x = x - self.vocab_start_idx
            x[mask] = 0
        y = F.embedding(x, self.weight)
        if world_size > 1:
            # 只保留当前进程负责的部分
            y[mask] = 0
            # 将每个进程的张量进行求和，并将结果广播给所有进程
            dist.all_reduce(y)
        return y

In [7]:
def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None) -> torch.Tensor:
    """自定义的线性运行,用于支持量化的相关操作"""
    # 标准精度运算
    # > 1：表示权重是标准精度（如float32等）
    # == 1：表示权重是量化权重（如int8等）
    if weight.element_size() > 1:
        return F.linear(x, weight, bias)
    # bfloat16反量化运算
    elif gemm_impl == "bf16":
        weight = weight_dequant(weight, weight.scale)
        return F.linear(x, weight, bias)
    # FP8量化运算
    else:
        # 量化时，同一个Block内的变量共享同一个缩放因子scale
        x, scale = act_quant(x, block_size)
        # 使用FP8量化的矩阵乘法
        y = fp8_gemm(x, scale, weight, weight.scale)
        # 偏置项不量化
        if bias is not None:
            y += bias
        return y


In [8]:
class Linear(nn.Module):
    """
    自定义线性层, 用于支持量化操作

    线性层nn.Linear(in_features, out_features),
    对应的权重shape=(out_features, in_features)
    线性层的行row和列column:
    行(第0维) = 输出维度 out_features
    列(第1维) = 输入维度 in_features

    与嵌入层对比：
    词嵌入矩阵通常是 (vocab_size, embed_dim),
    它的“行”索引的是 token,“列”是嵌入维度,不同于线性层的含义。
    """

    dtype = torch.bfloat16

    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype or Linear.dtype))
        # 若量化权重，则需要存储量化的缩放因子scale
        if self.weight.element_size() == 1:
            scale_out_features = (out_features + block_size - 1) // block_size
            scale_in_features = (in_features + block_size - 1) // block_size
            # 存储量化后的缩放因子scale（动态添加作为weight的属性）
            # 双挂载同时满足注册与使用
            self.weight.scale = self.scale = nn.Parameter(torch.empty(scale_out_features, scale_in_features, dtype=torch.float32))
        else:
            # 将scale注册为模块参数，方便管理
            self.register_parameter("scale", None)
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_paramter("bias", None)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return linear(x, self.weight, self.bias)
    

In [9]:
class ColumnParallelLinear(Linear):
    """列并行的线性层"""
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        assert out_features % world_size == 0, f"Output features must be divisible by world size (world_size={world_size})"
        self.part_out_features = out_features // world_size
        super().__init__(in_features, self.part_out_features, bias, dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = linear(x, self.weight, self.bias)
        return y

In [10]:
class RowParallelLinear(Linear):
    """行并行的线性层"""
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        assert in_features % world_size == 0, f"Input features must be divisible by world size (world_size={world_size})"
        self.part_in_features = in_features // world_size
        super().__init__(self.part_in_features, out_features, bias, dtype)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        行并行,每个rank: 
        x_r 形状 [B, in_features/world_size],
        W_r 形状 [out_features, in_features/world_size],
        局部 y_r 形状 [B, out_features], 得到的只是“部分贡献”,
        需要 all_reduce(sum) 聚合。
        """
        y = linear(x, self.weight)
        if world_size > 1:
            dist.all_reduce(y)
        if self.bias is not None:
            y += self.bias
        return y

In [11]:
class RMSNorm(nn.Module):
    """RMSNorm归一化"""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x: torch.Tensor):
        return F.rms_norm(x, (self.dim,), self.weight, self.eps)

In [12]:
# RoPE
def precompute_freqs_cis(args: ModelArgs) -> torch.Tensor:
    """预计算RoPE的频率和余弦/正弦值"""
    dim = args.qk_rope_head_dim  # 每个注意力头上应用 RoPE 的维度，必须为偶数
    seqlen = args.max_seq_len  # 最大序列长度
    beta_fast = args.beta_fast  # 控制在哪个“维度区间”逐步把频率从原始值过渡到缩放值（YaRN/NTK 风格的平滑外推）
    beta_slow = args.beta_slow  
    base = args.rope_theta  # 频率基数
    factor = args.rope_factor  # 长上下文外推时的频率缩放因子

    def find_correction_dim(num_rotations, dim, base, max_seq_len):
        """找到满足条件的修正维度"""
        return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))
    
    def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
        """找到满足条件的修正范围"""
        low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))
        high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))
        return max(low, 0), min(high, dim-1)
    
    def linear_ramp_factor(min, max, dim):
        """线性插值因子"""
        if min == max:
            max += 0.001
        linear_func = (torch.arange(dim, dtype=torch.float32) - min) / (max - min)
        ramp_func = torch.clamp(linear_func, 0, 1)
        return ramp_func

    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
    # 长上下文外推修正
    # 在维度轴上指定一个区间，把频率从原值平滑过渡到“缩小 factor 倍”的值，以获得更长的可用上下文。
    if seqlen > args.original_seq_len:
        # 给定希望在 max_seq_len 内完成 rotations 圈旋转，计算应对应到的维度索引
        low, high = find_correction_range(beta_fast, beta_slow, dim, base, args.original_seq_len)
        # 产生一个从低到高线性增长（0→1）的权重
        smooth = 1 - linear_ramp_factor(low, high, dim // 2)
        freqs = freqs / factor * (1 - smooth) + freqs * smooth

    # 生成按位置的相位与复指数
    t = torch.arange(seqlen)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis

def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """应用RoPE旋转嵌入"""
    # 使用复数时临时转为 float32，最后再转回原始 dtype
    dtype = x.dtype
    # 把最后一维两两组合为复数
    x = torch.view_as_complex(x.float().view(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))
    # 广播相乘实现“按位置旋转”， [1, T, 1, D/2] --> [B, T, H, D/2]
    # 等价于对每个位置施加旋转角
    y = torch.view_as_read(x * freqs_cis).flatten(3)
    return y.to(dtype)

In [13]:
class MLA(nn.Module):
    """MLA层"""
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.dim = args.dim  # 隐藏维度
        self.n_heads = args.n_heads  # 总的注意力头数
        self.n_local_heads = args.n_heads // world_size  # 并行时，每个rank负责的注意力头数
        self.q_lora_rank = args.q_lora_rank  # 作用于 Q 投影的 LoRA 低秩
        self.kv_lora_rank = args.kv_lora_rank  # 作用于 K/V 投影的 LoRA 低秩
        self.qk_nope_head_dim = args.qk_nope_head_dim  # 每个头在 Q/K 中“不做位置旋转”的那部分维度
        self.qk_rope_head_dim = args.qk_rope_head_dim  # 每个头在 Q/K 中“应用 RoPE”的那部分维度
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim  # 每个头的 Q/K 总维度
        self.v_head_dim = args.v_head_dim  # 每个头的 V 维度，一般qk_head_dim == v_head_dim == dim / n_heads，但此实现允许它们不同

        # 普通的（不使用q压缩）wq运算
        if self.q_lora_rank == 0:
            # 把 n_heads * qk_head_dim 按 rank 切分，让每个 rank 只负责 n_local_heads 个头的 Q
            # 减少和推迟跨rank的通信开销
            # 若使用按row的并行，由于只得到部分和，较早的时候就需要使用all-reduce才能得到完整的Q
            self.wq = ColumnParallelLinear(self.dim, self.n_heads * self.qk_head_dim)
        # 使用q压缩，把q投影到低秩空间
        else:
            self.wq_q = Linear(self.dim, self.q_lora_rank)
            self.q_norm = RMSNorm(self.q_lora_rank)
            self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)
        # 把向量投到两段拼接的空间（将原本的两个线性层，进行了拼接）
        # 前半段： kv_lora_rank 个维度，用于压缩kv
        # 后半段： qk_rope_head_dim 个维度，不经过压缩，直接用于应用RoPE
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)
        # 只对低秩分支做归一化
        self.kv_norm = RMSNorm(self.kv_lora_rank)
        # 恢复kv, 压缩的kv采用nope(不使用rope)，因此恢复后的维度为qk_nope_head_dim
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))
        # 注意力的输出投影
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)
        # 计算 softmax 的缩放因子
        self.softmax_scale = self.qk_head_dim ** -0.5
        # 长上下文外推修正
        if args.max_seq_len > args.original_seq_len:
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0
            self.softmax_scale = self.softmax_scale * mscale * mscale

        # 根据是否使用低秩压缩和“吸收”特性，采用不同的缓存策略
        # 分支一：普通的kv cache
        if attn_impl == "naive":
            self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.qk_head_dim), persistent=False)
            self.register_buffer("v_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.v_head_dim), persistent=False)
        # 分支二：使用低秩压缩的kv cache
        else:
            self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)
            self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """MLA层的前向计算, start_pos表示已缓存的token数"""
        bsz, seqlen, _ = x.size()
        end_pos = start_pos + seqlen
        # 根据是否使用低秩压缩，采用不同的计算方式q
        if self.q_lora_rank == 0:
            q = self.wq(x)
        else:
            q = self.wq_b(self.q_norm(self.wq_a(x)))
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)
        # 将q拆分为两段，前半段q_nope不使用rope，后半段q_pe使用rope
        # q_nope shape为 [bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim]
        # q_pe shape为   [bsz, seqlen, self.n_local_heads, self.qk_rope_head_dim]
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)
        # 对q_pe应用RoPE
        q_pe = apply_rotary_emb(q_pe, freqs_cis)
        
        # 将x投到两段拼接的空间
        kv = self.wkv_a(x)
        # 将kv拆分为两段，前半段为压缩后的kv，不使用rope；后半段k_pe，不压缩，使用rope
        # kv shape为   [bsz, seqlen, self.kv_lora_rank + self.qk_rope_head_dim]
        # k_pe shape为 [bsz, seqlen, self.qk_rope_head_dim]
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        # 将k_pe扩展维度并应用RoPE
        # 扩展维度后，shape为[bsz, seqlen, 1, self.qk_rope_head_dim]
        # 多个heads共用k_pe，类似MQA
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)
        # 根据是否使用低秩压缩和“吸收”特性，采用不同的计算方式注意力
        # 分支一：普通的注意力
        if attn_impl == "naive":
            q = torch.cat([q_nope, q_pe], dim=-1)
            # 恢复kv
            kv = self.wkv_b(self.kv_norm(kv))
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)
            # 拆分k、v
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)
            # 拼接k_nope和k_pe
            # 扩展维度后，shape为[bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.qk_rope_head_dim]
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)
            self.k_cache[:bsz, start_pos:end_pos] = k
            self.v_cache[:bsz, start_pos:end_pos] = v
            scores = torch.einsum("bahd,bthd->bsht", q, self.k_cache[:bsz, :end_pos]) * self.softmax_scale
        # 分支二：使用低秩压缩和“吸收”特性
        else:
            # 如果有scale，则使用weight_dequant反量化
            wkv_b = self.wkv_b.weight if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size)
            # 将wkv_b的shape
            #       [self.n_local_heads * (self.qk_nope_head_dim + self.v_head_dim), self.kv_lora_rank]
            # 调整为[self.n_local_heads, (self.qk_nope_head_dim + self.v_head_dim), self.kv_lora_rank]
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)
            # 将q_nope投到低秩空间，对应论文中提到的将wkv_b“吸收”进wq
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            self.kv_cache[:bsz, start_pos:end_pos] = self.kv_norm(kv)
            self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)
            # 分别计算q_nope和q_pe的注意力分数
            # 前者在低秩空间，后者在原始空间
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) + 
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale
        # 就用mask(-inf)来屏蔽未来位置
        if mask is not None:
            scores += mask.unsqueeze(1)
        # 归一化
        # shape = [bsz, seqlen, self.n_local_heads, total_len]
        scores = scores.softmax(dim=-1, dtype=torch.float32).type_as(x)
        # 分支一：计算上下文向量
        if attn_impl == "naive":
            x = torch.einsum("bsht,bthd->bshd", scores, self.v_cache[:bsz, :end_pos])
        # 分支二：采用absorb特性
        else:
            # 用注意力权重对低秩缓存加权，得到低秩上下文
            # shape = [bsz, seqlen, self.n_local_heads, self.kv_lora_rank]
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])
            # 用反投影矩阵把低秩上下文映射回原维度
            # shape = [bsz, seqlen, self.n_local_heads, self.v_head_dim]
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])
        x = self.wo(x.flatten(2))
        return x

In [ ]:
class MLP(nn.Module):
    """MLP(FFN)模块"""
    def __init__(self, dim: int, inter_dim: int):
        super().__init__()
        self.w1 = ColumnParallelLinear(dim, inter_dim)
        self.w2 = RowParallelLinear(inter_dim, dim)
        self.w3 = ColumnParallelLinear(dim, inter_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
class Gate(nn.Module):
    """门控模块(MoE)"""
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.dim = args.dim
        self.topk = args.n_activated_experts
        self.n_groups = args.n_expert_groups
        self.topk_groups = args.n_limited_groups
        self.score_func = args.score_func
        self.route_scale = args.route_scale
        self.weight = nn.Parameter(torch.empty(args.n_routed_experts, args.dim))
        self.bias = nn.Parameter(torch.empty(args.n_routed_experts)) if self.dim == 7168 else None

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # scores shape = [bsz * seqlen, n_routed_experts]
        scores = linear(x, self.weight)
        if self.score_func == "softmax":
            scores = scores.softmax(dim=-1, dtype=torch.float32)
        else:
            scores = scores.sigmoid()
        original_scores = scores
        if self.bias is not None:
            scores = scores + self.bias
        # 专家分组
        if self.n_groups > 1:
            # 将scores reshape为[bsz * seqlen, n_groups, n_per_group]
            scores = scores.view(x.size(0), self.n_groups, -1)
            # 计算每个组的得分
            # 两种取法，应该是为了于训练对齐，仅dim=7168的模型使用了top2的组得分
            if self.bias is None:
                group_scores = scores.amax(dim=-1)
            else:
                group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)
            # mask非topk_groups的scores
            indices = group_scores.topk(self.topk_groups, dim=-1)[1]
            mask = scores.new_ones(x.size(0), self.n_groups, dtype=bool).scatter_(1, indices, False)
            # scores shape = [bsz * seqlen, n_routed_experts]
            scores = scores.masked_fill(mask.unsqueeze(-1), -float("inf")).flatten(1)
        # 选出topk个专家
        indices = torch.topk(scores, self.topk, dim=-1)[1]
        # 取出对应的权重
        # shape = [bsz * seqlen, topk]
        weights = original_scores.gather(1, indices)
        if self.score_func == "softmax":
            # 重新归一化
            weights /= weights.sum(dim=-1, keepdim=True)
        weights *= self.route_scale
        return weights.type_as(x), indices

In [ ]:
class Expert(nn.Module):
    """专家模块(与MLP类似，区别是单个expert不支持并行)"""
    def __init__(self, dim: int, inter_dim: int):
        super().__init__()
        self.w1 = Linear(dim, inter_dim)
        self.w2 = Linear(inter_dim, dim)
        self.w3 = Linear(dim, inter_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
class MoE(nn.Module):
    """MoE模块"""
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.dim = args.dim
        assert args.n_routed_experts % world_size == 0, f"Number of experts must be divisible by world size (world_size={world_size})"
        self.n_routed_experts = args.n_routed_experts
        self.n_local_experts = args.n_routed_experts // world_size
        self.n_activated_experts = args.n_activated_experts
        self.experts_start_idx = rank * self.n_local_experts
        self.experts_end_idx = self.experts_start_idx + self.n_local_experts
        self.gate = Gate(args)
        self.experts = nn.ModuleList([Expert(args.dim, args.moe_inter_dim) if self.experts_start_idx <= i < self.experts_end_idx else None
                                      for i in range(self.n_routed_experts)])
        self.shared_experts = MLP(args.dim, args.n_shared_experts * args.moe_inter_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shape = x.size()
        # reshape后 = [bsz * seqlen, dim]
        x = x.view(-1, self.dim)
        # 选出topk个专家[bsz * seqlen, topk]
        weights, indices = self.gate(x)
        y = torch.zeros_like(x)
        # 统计各个专家被选中的次数
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts).tolist()
        for i in range(self.experts_start_idx, self.experts_end_idx):
            if counts[i] == 0:
                continue
            expert = self.experts[i]
            # 选出选中当前专家的tokens，完成前向计算和加权求和
            idx, top = torch.where(indices == i)
            # None用于扩展维度，使得expert的输出与weights的维度匹配[bsz * seqlen, 1]，用于广播
            y[idx] += expert(x[idx]) * weights[idx, top, None]
        # 共享专家前向计算
        z = self.shared_experts(y)
        # 将路由专家的输出进行归约
        if world_size > 1:
            dist.all_reduce(y)
        # 将路由专家和共享专家的输出相加，并恢复原shape = [bsz, seqlen, dim]
        return (y + z).view(shape)
        

In [ ]:
class Block(nn.Module):
    """Transformer Block, 仅包含attn和ffn"""
    def __init__(self, layer_id: int, args: ModelArgs):
        super().__init__()
        self.attn = MLA(args)
        self.ffn = MLP(args.dim, args.inter_dim) if layer_id < args.n_dense_layers else MoE(args)
        self.attn_norm = RMSNorm(args.dim)
        self.ffn_norm = RMSNorm(args.dim)
    
    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # 带快捷连接的attn和ffn
        x = x + self.attn(self.attn_norm(x), start_pos, freqs_cis, mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x

In [ ]:
class Transformer(nn.Module):
    """完整的DeepSeek-V3模型"""
    def __init__(self, args: ModelArgs):
        global world_size, rank
        world_size = dist.get_world_size() if dist.is_initialized() else 1
        rank = dist.get_rank() if dist.is_initialized() else 0
        Linear.dtype = torch.float8_e4m3fn if args.dtype == "fp8" else torch.bfloat16
        super().__init__()
        self.max_seq_len = args.max_seq_len
        self.embed = ParallelEmbedding(args.vocab_size, args.dim)
        self.layers = torch.nn.ModuleList()
        for layer_id in range(args.n_layers):
            self.layers.append(Block(layer_id, args))
        self.norm = RMSNorm(args.dim)
        self.head = ColumnParallelLinear(args.dim, args.vocab_size, dtype=torch.get_default_etype())
        self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)
    
    # 用于开启“推理模式”。它比 no_grad 更彻底，专为纯推理优化。
    @torch.inference_mode()
    def forward(self, tokens: torch.Tensor, start_pos: int = 0):
        seqlen = tokens.size(1)
        h = self.embed(tokens)
        freqs_cis = self.freqs_cis[start_pos:start_pos + seqlen]
        mask = None
        if seqlen > 1:
            # 原地取严格上三角（主对角线以上，k=1）的mask
            mask = torch.full((seqlen, seqlen), float("-inf"), device=tokens.device).triu_(1)
        for layer in self.layers:
            h = layer(h, start_pos, freqs_cis, mask)
        # out poojection
        h = self.norm(h)[:, -1]
        logits = self.head(h)
        if world_size > 1:
            all_logits = [torch.empty_like(logits) for _ in range(world_size)]
            dist.all_gather(all_logits, logits)
            logits = torch.cat(all_logits, dim=-1)
        return logits

if __name__ == "__main__":
    torch.set_default_dtype(torch.bfloat16)
    torch.set_default_device("cuda")
    torch.manual_seed(0)
    args = ModelArgs()
    # 随机生成两个长度为128的token序列
    x = torch.randint(0, args.vocab_size, (2, 128))
    model = Transformer(args)
    print(model(x).size())

